# Streaming ML Framework — Demo Notebook

## Framework Overview

This framework is a **pure-NumPy** streaming machine-learning library designed for
scenarios where data arrives in batches (chunks) over time.
It has **no dependency** on scikit-learn, scipy, or any other third-party ML library —
only `numpy` and `matplotlib` are used.

| Module | Purpose |
|---|---|
| `io.py` | Custom CSV read/write, streaming generator, chunk splitter |
| `preprocessing.py` | `StandardScaler`, `MinMaxScaler`, `Imputer` — all support `partial_fit` |
| `stats.py` | Streaming statistics (`StreamStats`, `chunk_mean/variance/quantile/histogram`) |
| `tree.py` | Decision tree classifier (Gini / Entropy, supports `partial_fit`) |
| `ensemble.py` | `EnsembleClassifier` (Bagging / Random Forest) with streaming support |
| `metrics.py` | Streaming metrics (`Accuracy`, `F1Score`, `ConfusionMatrix`) and batch functions |
| `pipeline.py` | `Pipeline`: chains transformers + estimator, supports `partial_fit` |
| `stream.py` | `StreamTrainer`: iterates chunks, logs metrics, tracks memory |
| `visualise.py` | Plotting tools (metric trends, model comparison, scatter, confusion matrix) |

### Four Core Requirements Covered in This Demo

1. **Load a dataset from a CSV file using `io.py`**
2. **Split the dataset into multiple chunks to simulate a streaming data setting**
3. **Train the pipeline incrementally by calling `.partial_fit()` on each chunk**
4. **Log and visualise key metrics over time using `visualise.py`**

## Cell 1 — Imports & Dataset Loading (Core Requirement 1)

Import all framework modules along with `numpy` and `matplotlib`.

A synthetic binary-classification dataset is generated with NumPy (1000 samples × 6 features;
the first three features are informative, the last three are noise).
The data is written to CSV with `io.save_csv` and read back with `io.load_csv`,
satisfying **Core Requirement 1**.

In [ ]:
# Cell 1 — Imports, generate synthetic dataset, save with io.save_csv, load with io.load_csv (Core Req 1)
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from framework.io import load_csv, save_csv, split_into_chunks
from framework.preprocessing import StandardScaler
from framework.ensemble import RandomForestClassifier, EnsembleClassifier
from framework.pipeline import Pipeline
from framework.stream import StreamTrainer
from framework.metrics import Accuracy, F1Score, confusion_matrix as compute_cm
from framework.visualise import (
    plot_metric_over_time, compare_models,
    plot_predictions_vs_ground_truth, plot_confusion_matrix,
)

# Generate synthetic dataset (1000 samples x 6 features, binary classification)
rng = np.random.default_rng(seed=0)
N, D = 1000, 6
X_raw = rng.normal(loc=0.0, scale=2.0, size=(N, D))
weights = np.array([1.5, -1.0, 0.8, 0.0, 0.0, 0.0])
y_raw = (X_raw @ weights > 0).astype(int)

# Core Req 1: write CSV with io.save_csv, then read back with io.load_csv
csv_path = '/tmp/demo_data.csv'
headers = [f'f{i}' for i in range(D)] + ['label']
save_csv(csv_path, np.column_stack([X_raw, y_raw]), headers=headers)

data, cols = load_csv(csv_path, has_header=True)
X = data[:, :-1]
y = data[:, -1].astype(int)

print(f'Dataset: {X.shape}, class distribution: {np.bincount(y)}')
print(f'Columns: {cols}')

## Cell 2 — Chunk Splitting & Pipeline Construction (Core Requirement 2)

`split_into_chunks` (`io.py`) divides the 1000-sample dataset into **10 consecutive chunks**,
simulating a streaming scenario where data arrives batch by batch — satisfying **Core Requirement 2**.

Two pipelines are also constructed here for later comparison:
- **Random Forest**: `StandardScaler` → `RandomForestClassifier` (√d feature sampling)
- **Bagging**: `StandardScaler` → `EnsembleClassifier` (all features, bootstrap sampling)

In [ ]:
# Cell 2 — Split chunks to simulate streaming (Core Req 2) and build two Pipelines

# Core Req 2: split into 10 chunks with split_into_chunks
N_CHUNKS = 10
chunks = split_into_chunks(X, y, n_chunks=N_CHUNKS)
print(f'Split into {N_CHUNKS} chunks, {N // N_CHUNKS} samples each')

# Build Random Forest and Bagging pipelines
def make_rf_pipeline(seed=42):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=10, max_depth=5, random_state=seed)),
    ])

def make_bag_pipeline(seed=42):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', EnsembleClassifier(n_estimators=10, method='bagging', max_depth=5, random_state=seed)),
    ])

print('RF Pipeline  : StandardScaler -> RandomForestClassifier(n=10, depth=5)')
print('Bag Pipeline : StandardScaler -> EnsembleClassifier(bagging, n=10, depth=5)')

## Cell 3 — Incremental Training (Core Requirement 3)

`StreamTrainer` calls `pipeline.partial_fit(X_chunk, y_chunk)` on every chunk,
enabling true online learning — model state accumulates incrementally without resetting,
satisfying **Core Requirement 3**.

> **Metric note**: `Accuracy` and `F1Score` are **cumulative** — they reflect
> the model's overall performance across all chunks seen so far,
> not the instantaneous score on the current chunk alone.

In [ ]:
# Cell 3 — Incremental training: call .partial_fit() on each chunk (Core Req 3)
# StreamTrainer internally calls pipeline.partial_fit(X_chunk, y_chunk).
# Metrics are cumulative — they reflect overall performance across all data seen so far.

rf_trainer = StreamTrainer(
    pipeline=make_rf_pipeline(seed=42),
    metrics=[Accuracy(), F1Score()],
    log_memory=True,
)
bag_trainer = StreamTrainer(
    pipeline=make_bag_pipeline(seed=42),
    metrics=[Accuracy(), F1Score()],
    log_memory=True,
)

print('Chunk | RF Acc  RF F1  | Bag Acc Bag F1')
print('-' * 45)
for i, (Xc, yc) in enumerate(chunks):
    rf  = rf_trainer.fit_chunk(Xc, yc)
    bag = bag_trainer.fit_chunk(Xc, yc)
    print(f'  {i:2d}  | {rf["accuracy"]:.4f}  {rf["f1score"]:.4f}  | {bag["accuracy"]:.4f}  {bag["f1score"]:.4f}')

rf_log  = rf_trainer.get_log()
bag_log = bag_trainer.get_log()
rf_acc  = [r['accuracy'] for r in rf_log]
rf_f1   = [r['f1score']  for r in rf_log]
bag_acc = [r['accuracy'] for r in bag_log]
bag_f1  = [r['f1score']  for r in bag_log]

## Cell 4 — Metric Trend Plots (Core Requirement 4)

`plot_metric_over_time` (`visualise.py`) plots the Random Forest's cumulative
**Accuracy**, **Error Rate**, and **F1 Score** as training progresses through chunks.

Expected observation: accuracy rises and error rate falls as more data accumulates,
demonstrating the benefit of streaming / incremental learning.

In [ ]:
# Cell 4 — Accuracy, Error Rate, F1 trend plots (Core Req 4)
# Plot cumulative Accuracy, Error Rate, and F1 for the RF model across chunks.

rf_error = [1.0 - a for a in rf_acc]

plot_metric_over_time(
    rf_acc, title='RF — Cumulative Accuracy over Chunks',
    ylabel='Accuracy', save_path='/tmp/rf_accuracy.png',
)
plt.show()

plot_metric_over_time(
    rf_error, title='RF — Cumulative Error Rate over Chunks',
    ylabel='Error Rate (1 - Accuracy)', save_path='/tmp/rf_error.png',
)
plt.show()

plot_metric_over_time(
    rf_f1, title='RF — Cumulative F1 Score over Chunks',
    ylabel='F1 Score', save_path='/tmp/rf_f1.png',
)
plt.show()

print(f'Accuracy  : {rf_acc[0]:.4f} -> {rf_acc[-1]:.4f}')
print(f'Error Rate: {rf_error[0]:.4f} -> {rf_error[-1]:.4f}')
print(f'F1 Score  : {rf_f1[0]:.4f} -> {rf_f1[-1]:.4f}')

## Cell 5 — Model Comparison (Core Requirement 4)

`compare_models` (`visualise.py`) overlays the Accuracy and F1 curves of
**Random Forest** and **Bagging** on the same axes for direct visual comparison.

The key difference is feature sampling: Random Forest uses only √d features per tree,
while Bagging uses all features.

In [ ]:
# Cell 5 — Model comparison: RF vs Bagging (Core Req 4)
# Overlay Accuracy and F1 curves of both models on the same axes.

compare_models(
    rf_acc, bag_acc,
    labels=['Random Forest', 'Bagging'],
    title='Model Comparison — Cumulative Accuracy',
    ylabel='Accuracy',
    save_path='/tmp/model_comparison_acc.png',
)
plt.show()

compare_models(
    rf_f1, bag_f1,
    labels=['Random Forest', 'Bagging'],
    title='Model Comparison — Cumulative F1 Score',
    ylabel='F1 Score',
    save_path='/tmp/model_comparison_f1.png',
)
plt.show()

print(f'RF  final: Accuracy={rf_acc[-1]:.4f}, F1={rf_f1[-1]:.4f}')
print(f'Bag final: Accuracy={bag_acc[-1]:.4f}, F1={bag_f1[-1]:.4f}')

## Cell 6 — Prediction Scatter & Confusion Matrix (Core Requirement 4)

The **last chunk** is used to visualise RF predictions in two ways:
- `plot_predictions_vs_ground_truth`: compares predicted vs. true label for each sample
- `plot_confusion_matrix`: heatmap of TN / FP / FN / TP counts;
  colour intensity represents magnitude

In [ ]:
# Cell 6 — Prediction scatter plot & confusion matrix (Core Req 4)
# Visualise RF predictions and confusion matrix on the last chunk.

Xlast, ylast = chunks[-1]
y_pred_last = rf_trainer.pipeline.predict(Xlast)

plot_predictions_vs_ground_truth(
    ylast, y_pred_last,
    title='RF Predictions vs Ground Truth — Last Chunk',
    save_path='/tmp/pred_vs_true.png',
)
plt.show()

cm = compute_cm(ylast, y_pred_last)
plot_confusion_matrix(
    cm, class_names=['Class 0', 'Class 1'],
    save_path='/tmp/confusion_matrix.png',
)
plt.show()

n_correct = int(np.sum(y_pred_last == ylast))
print(f'Last chunk: {n_correct}/{len(ylast)} correct, accuracy={n_correct/len(ylast):.4f}')
print(f'Confusion matrix: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}')

## Cell 7 — Memory Tracking & Final Summary (Core Requirement 4)

`plot_metric_over_time` tracks the RSS memory usage (MB) of the RF pipeline
across all training chunks.

One key advantage of streaming learning is that memory should remain stable
rather than growing linearly with dataset size.
A final summary table confirms all four core requirements have been met.

In [ ]:
# Cell 7 — Memory usage tracking & final summary (Core Req 4)

mem_mb = [r.get('memory_mb', 0.0) for r in rf_log]
plot_metric_over_time(
    mem_mb,
    title='RF Pipeline — Memory Usage (RSS) over Chunks',
    ylabel='RSS Memory (MB)',
    save_path='/tmp/memory_usage.png',
)
plt.show()
print(f'Memory: initial={mem_mb[0]:.1f} MB, peak={max(mem_mb):.1f} MB\n')

print('=' * 48)
print('        Streaming Training — Final Summary')
print('=' * 48)
print(f'  Dataset : {N} samples x {D} features, binary classification')
print(f'  Chunks  : {N_CHUNKS} chunks, {N // N_CHUNKS} samples each')
print()
header = f"  {'Model':<18} {'Accuracy':>10} {'F1 Score':>10}"
print(header)
print(f'  {"-"*40}')
print(f'  {"Random Forest":<18} {rf_acc[-1]:>10.4f} {rf_f1[-1]:>10.4f}')
print(f'  {"Bagging":<18} {bag_acc[-1]:>10.4f} {bag_f1[-1]:>10.4f}')
print()
print('  Core Requirements:')
print('  1. io.load_csv — loaded dataset from CSV')
print('  2. split_into_chunks — split into 10 chunks (streaming simulation)')
print('  3. pipeline.partial_fit() — incremental training on each chunk')
print('  4. visualise.py — Accuracy / Error / F1 / model comparison / scatter / confusion matrix / memory')